# Movie Graph Database 구축 튜토리얼

이 노트북은 `graph_build.py`를 단계별로 실행하면서 Neo4j에 영화 그래프 데이터베이스를 구축하는 교육용 튜토리얼입니다.

## 그래프 스키마 개요

```
                    ┌─────────┐
                    │  Genre  │
                    └────▲────┘
                         │ HAS_GENRE
┌──────────────┐    ┌────┴────┐    ┌───────────────────┐
│   Country    │◄───│  Movie  │───►│ProductionCompany  │
└──────────────┘    └────┬────┘    └───────────────────┘
  PRODUCED_IN            │ PRODUCED_BY
                         │
┌──────────────┐         │         ┌───────────────────┐
│SpokenLanguage│◄────────┘         │      Person       │
└──────────────┘  HAS_LANGUAGE     │  (Actor/Director/ │
                                   │  Producer/User)   │
                                   └───────────────────┘
                                   ACTED_IN / DIRECTED /
                                   PRODUCED / RATED
```

## CSV 파일 개요

| 파일명 | 주요 컬럼 | 행 수 | 용도 |
|--------|----------|-------|------|
| normalized_movies.csv | tmdbId, title, budget, revenue, overview... | 45,572 | 영화 메타데이터 |
| normalized_cast.csv | actor_id, name, character, tmdbId | 559,733 | 배우 정보 |
| normalized_crew.csv | crew_id, name, job, tmdbId | 40,072 | 감독/프로듀서 |
| normalized_genres.csv | genre_id, genre_name, tmdbId | 91,106 | 장르 |
| normalized_production_companies.csv | company_id, company_name, tmdbId | 70,545 | 제작사 |
| normalized_production_countries.csv | country_code, country_name, tmdbId | 49,423 | 제작 국가 |
| normalized_spoken_languages.csv | language_code, language_name, tmdbId | 53,300 | 사용 언어 |
| normalized_keywords.csv | tmdbId, keywords | 45,432 | 키워드 |
| normalized_links.csv | movieId, imdbId, tmdbId | 45,843 | ID 매핑 |
| normalized_ratings_small.csv | userId, movieId, rating, timestamp | 100,004 | 사용자 평점 |

---
## Step 0. 패키지 설치

처음 실행하는 경우 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install neo4j python-dotenv pandas -q

---
## Step 1. 환경 변수 설정

프로젝트 루트의 `.env` 파일에 아래 항목이 채워져 있어야 합니다.

```
NEO4J_URI=bolt+s://<your-neo4j-uri>
NEO4J_USER=neo4j
NEO4J_PASSWORD=<your-password>
NEO4J_DATABASE=neo4j
```

`.env` 파일이 없다면 `example.env`를 복사해 작성하세요.

In [76]:
import os
import warnings
import pandas as pd
from neo4j import GraphDatabase
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USER     = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

print(f"URI     : {NEO4J_URI}")
print(f"User    : {NEO4J_USER}")
print(f"Database: {NEO4J_DATABASE}")

URI     : neo4j://127.0.0.1:7687
User    : neo4j
Database: neo4j


---
## Step 2. Neo4j 드라이버 연결 & 연결 테스트

`GraphDatabase.driver()`로 드라이버를 생성하고, 간단한 Cypher로 연결 상태를 확인합니다.

In [77]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    database=NEO4J_DATABASE
)

# 연결 테스트
with driver.session() as session:
    result = session.run("RETURN 'Neo4j 연결 성공!' AS message")
    print(result.single()["message"])

Neo4j 연결 성공!


---
## Step 3. CreateGraph 클래스 정의

`graph_build.py`의 `CreateGraph` 클래스를 그대로 노트북에 올립니다.  
각 메서드는 이후 단계에서 하나씩 호출합니다.

In [78]:
class CreateGraph:
    """Neo4j 영화 그래프 데이터베이스 구축 클래스.

    외부에서 생성한 driver를 받아 재사용합니다.
    드라이버 생명주기(close)는 호출자가 관리합니다.
    """

    def __init__(self, driver):
        # 새 연결을 만들지 않고 노트북의 driver를 공유합니다.
        self.driver = driver

    @staticmethod
    def _run_batched(session, query, rows, batch_size=5000):
        """rows 리스트를 batch_size 단위로 나눠 쿼리를 실행합니다."""
        for i in range(0, len(rows), batch_size):
            session.run(query, rows=rows[i:i + batch_size])

    # ── 초기화 ──────────────────────────────────────────────────────────
    def db_cleanup(self):
        print("데이터베이스를 초기화합니다...")
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
        print("초기화 완료. 빈 데이터베이스 준비됨.")

    def create_constraints_indexes(self):
        queries = [
            "CREATE CONSTRAINT unique_tmdb_id IF NOT EXISTS FOR (m:Movie) REQUIRE m.tmdbId IS UNIQUE;",
            "CREATE CONSTRAINT unique_movie_id IF NOT EXISTS FOR (m:Movie) REQUIRE m.movieId IS UNIQUE;",
            "CREATE CONSTRAINT unique_prod_id  IF NOT EXISTS FOR (p:ProductionCompany) REQUIRE p.company_id IS UNIQUE;",
            "CREATE CONSTRAINT unique_genre_id IF NOT EXISTS FOR (g:Genre) REQUIRE g.genre_id IS UNIQUE;",
            "CREATE CONSTRAINT unique_lang_id  IF NOT EXISTS FOR (l:SpokenLanguage) REQUIRE l.language_code IS UNIQUE;",
            "CREATE CONSTRAINT unique_country_id IF NOT EXISTS FOR (c:Country) REQUIRE c.country_code IS UNIQUE;",
            "CREATE INDEX actor_id IF NOT EXISTS FOR (p:Person) ON (p.actor_id);",
            "CREATE INDEX crew_id  IF NOT EXISTS FOR (p:Person) ON (p.crew_id);",
            "CREATE INDEX movieId  IF NOT EXISTS FOR (m:Movie)  ON (m.movieId);",
            "CREATE INDEX user_id  IF NOT EXISTS FOR (p:Person) ON (p.user_id);",
        ]
        with self.driver.session() as session:
            for q in queries:
                session.run(q)
        print("제약 조건 및 인덱스 생성 완료.")

    # ── 데이터 로드 ─────────────────────────────────────────────────────
    def load_movies(self, csv_path, limit):
        df = pd.read_csv(csv_path, nrows=limit)
        df['tmdbId'] = pd.to_numeric(df['tmdbId'], errors='coerce')
        df = df.dropna(subset=['tmdbId'])
        df['tmdbId'] = df['tmdbId'].astype(int)
        for col in ['title', 'original_title', 'original_language', 'tagline', 'overview',
                    'release_date', 'belongs_to_collection']:
            df[col] = df[col].fillna('None')
        df['adult']   = df['adult'].apply(lambda x: 'Yes' if str(x).strip() == '1' else 'No')
        df['budget']  = pd.to_numeric(df['budget'],  errors='coerce').fillna(0).astype(int)
        df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0).astype(int)
        df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce').fillna(0.0)

        query = """
        UNWIND $rows AS row
        MERGE (m:Movie {tmdbId: row.tmdbId})
        ON CREATE SET
            m.title                 = row.title,
            m.original_title        = row.original_title,
            m.adult                 = row.adult,
            m.budget                = row.budget,
            m.original_language     = row.original_language,
            m.revenue               = row.revenue,
            m.tagline               = row.tagline,
            m.overview              = row.overview,
            m.release_date          = row.release_date,
            m.runtime               = row.runtime,
            m.belongs_to_collection = row.belongs_to_collection
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"영화 로드 완료 ({len(rows):,}개): {csv_path}")

    def load_genres(self, csv_path):
        df = pd.read_csv(csv_path)
        df['tmdbId']   = pd.to_numeric(df['tmdbId'],   errors='coerce')
        df['genre_id'] = pd.to_numeric(df['genre_id'], errors='coerce')
        df = df.dropna(subset=['tmdbId', 'genre_id'])
        df['tmdbId']   = df['tmdbId'].astype(int)
        df['genre_id'] = df['genre_id'].astype(int)

        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (g:Genre {genre_id: row.genre_id})
          ON CREATE SET g.genre_name = row.genre_name
        MERGE (m)-[:HAS_GENRE]->(g)
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"장르 로드 완료 ({len(rows):,}행): {csv_path}")

    def load_production_companies(self, csv_path):
        df = pd.read_csv(csv_path)
        df['tmdbId']     = pd.to_numeric(df['tmdbId'],     errors='coerce')
        df['company_id'] = pd.to_numeric(df['company_id'], errors='coerce')
        df = df.dropna(subset=['tmdbId', 'company_id'])
        df['tmdbId']     = df['tmdbId'].astype(int)
        df['company_id'] = df['company_id'].astype(int)

        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (pc:ProductionCompany {company_id: row.company_id})
          ON CREATE SET pc.company_name = row.company_name
        MERGE (m)-[:PRODUCED_BY]->(pc)
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"제작사 로드 완료 ({len(rows):,}행): {csv_path}")

    def load_production_countries(self, csv_path):
        df = pd.read_csv(csv_path)
        df['tmdbId'] = pd.to_numeric(df['tmdbId'], errors='coerce')
        df = df.dropna(subset=['tmdbId', 'country_code'])
        df['tmdbId']       = df['tmdbId'].astype(int)
        df['country_code'] = df['country_code'].astype(str)

        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (c:Country {country_code: row.country_code})
          ON CREATE SET c.country_name = row.country_name
        MERGE (m)-[:PRODUCED_IN]->(c)
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"제작 국가 로드 완료 ({len(rows):,}행): {csv_path}")

    def load_spoken_languages(self, csv_path):
        df = pd.read_csv(csv_path)
        df['tmdbId'] = pd.to_numeric(df['tmdbId'], errors='coerce')
        df = df.dropna(subset=['tmdbId', 'language_code'])
        df['tmdbId']        = df['tmdbId'].astype(int)
        df['language_code'] = df['language_code'].astype(str)

        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (l:SpokenLanguage {language_code: row.language_code})
          ON CREATE SET l.language_name = row.language_name
        MERGE (m)-[:HAS_LANGUAGE]->(l)
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"사용 언어 로드 완료 ({len(rows):,}행): {csv_path}")

    def load_keywords(self, csv_path):
        df = pd.read_csv(csv_path)
        df['tmdbId'] = pd.to_numeric(df['tmdbId'], errors='coerce')
        df = df.dropna(subset=['tmdbId'])
        df['tmdbId']   = df['tmdbId'].astype(int)
        df['keywords'] = df['keywords'].fillna('')

        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        SET m.keywords = row.keywords
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"키워드 로드 완료 ({len(rows):,}행): {csv_path}")

    def load_person_actors(self, csv_path, chunk_size=20000):
        """대용량 cast CSV를 chunk 단위로 읽어 배우 노드와 ACTED_IN 관계를 생성합니다."""
        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (p:Person {actor_id: row.actor_id})
          ON CREATE SET p.name = row.name, p.role = 'actor'
        MERGE (p)-[a:ACTED_IN]->(m)
          ON CREATE SET a.character = row.character, a.cast_id = row.cast_id
        """
        total = 0
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
            chunk['tmdbId']   = pd.to_numeric(chunk['tmdbId'],   errors='coerce')
            chunk['actor_id'] = pd.to_numeric(chunk['actor_id'], errors='coerce')
            chunk['cast_id']  = pd.to_numeric(chunk['cast_id'],  errors='coerce').fillna(0)
            chunk = chunk.dropna(subset=['tmdbId', 'actor_id'])
            chunk['tmdbId']    = chunk['tmdbId'].astype(int)
            chunk['actor_id']  = chunk['actor_id'].astype(int)
            chunk['cast_id']   = chunk['cast_id'].astype(int)
            chunk['character'] = chunk['character'].fillna('None')
            rows = chunk.to_dict('records')
            with self.driver.session() as session:
                session.run(query, rows=rows)
            total += len(rows)
        print(f"배우 로드 완료 ({total:,}행): {csv_path}")

        with self.driver.session() as session:
            session.run("MATCH (n:Person) WHERE n.role='actor' SET n:Actor")
        print("Actor 레이블 추가 완료")

    def load_person_crew(self, csv_path):
        """감독·프로듀서를 job별로 나눠 DIRECTED/PRODUCED 관계를 생성합니다. (APOC 불필요)"""
        query_dir = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (p:Person {crew_id: row.crew_id})
          ON CREATE SET p.name = row.name, p.role = 'Director'
        MERGE (p)-[:DIRECTED]->(m)
        """
        query_prod = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        MERGE (p:Person {crew_id: row.crew_id})
          ON CREATE SET p.name = row.name, p.role = 'Producer'
        MERGE (p)-[:PRODUCED]->(m)
        """
        df = pd.read_csv(csv_path)
        df['tmdbId']  = pd.to_numeric(df['tmdbId'],  errors='coerce')
        df['crew_id'] = pd.to_numeric(df['crew_id'], errors='coerce')
        df = df.dropna(subset=['tmdbId', 'crew_id'])
        df['tmdbId']  = df['tmdbId'].astype(int)
        df['crew_id'] = df['crew_id'].astype(int)

        directors = df[df['job'] == 'Director'].to_dict('records')
        producers = df[df['job'] == 'Producer'].to_dict('records')

        with self.driver.session() as session:
            if directors:
                self._run_batched(session, query_dir, directors)
            if producers:
                self._run_batched(session, query_prod, producers)

        with self.driver.session() as session:
            session.run("MATCH (n:Person) WHERE n.role='Director' SET n:Director")
        with self.driver.session() as session:
            session.run("MATCH (n:Person) WHERE n.role='Producer' SET n:Producer")
        print(f"크루 로드 완료 (감독 {len(directors):,}, 프로듀서 {len(producers):,}): {csv_path}")

    def load_links(self, csv_path):
        df = pd.read_csv(csv_path)
        df['tmdbId']  = pd.to_numeric(df['tmdbId'],  errors='coerce')
        df['movieId'] = pd.to_numeric(df['movieId'], errors='coerce')
        df = df.dropna(subset=['tmdbId', 'movieId'])
        df['tmdbId']  = df['tmdbId'].astype(int)
        df['movieId'] = df['movieId'].astype(int)
        df['imdbId']  = df['imdbId'].astype(str)

        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {tmdbId: row.tmdbId})
        SET m.movieId = row.movieId, m.imdbId = row.imdbId
        """
        rows = df.to_dict('records')
        with self.driver.session() as session:
            self._run_batched(session, query, rows)
        print(f"링크 로드 완료 ({len(rows):,}행): {csv_path}")

    def load_ratings(self, csv_path, chunk_size=20000):
        query = """
        UNWIND $rows AS row
        MATCH (m:Movie {movieId: row.movieId})
        MERGE (p:Person {user_id: row.userId})
          ON CREATE SET p.role = 'user'
        MERGE (p)-[r:RATED]->(m)
          ON CREATE SET r.rating = row.rating, r.timestamp = row.timestamp
        """
        total = 0
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
            chunk['movieId']   = pd.to_numeric(chunk['movieId'],   errors='coerce')
            chunk['userId']    = pd.to_numeric(chunk['userId'],    errors='coerce')
            chunk['rating']    = pd.to_numeric(chunk['rating'],    errors='coerce')
            chunk['timestamp'] = pd.to_numeric(chunk['timestamp'], errors='coerce')
            chunk = chunk.dropna(subset=['movieId', 'userId', 'rating'])
            chunk['movieId']   = chunk['movieId'].astype(int)
            chunk['userId']    = chunk['userId'].astype(int)
            chunk['timestamp'] = chunk['timestamp'].fillna(0).astype(int)
            rows = chunk.to_dict('records')
            with self.driver.session() as session:
                session.run(query, rows=rows)
            total += len(rows)
        print(f"평점 로드 완료 ({total:,}행): {csv_path}")

        with self.driver.session() as session:
            session.run("MATCH (n:Person) WHERE n.role='user' SET n:User")
        print("User 레이블 추가 완료")

print("CreateGraph 클래스 정의 완료")

CreateGraph 클래스 정의 완료


---
## Cypher 핵심 문법 가이드

이 튜토리얼의 로드 함수들에서 반복적으로 등장하는 Cypher 구문을 정리합니다.

---

### 1. `UNWIND` — 리스트를 행으로 펼치기

Python에서 전달한 리스트(`$rows`)를 한 행씩 처리합니다.  
`LOAD CSV` 없이 Python 데이터를 Neo4j로 전달할 때 핵심 구문입니다.

```cypher
UNWIND $rows AS row
MERGE (m:Movie {tmdbId: row.tmdbId})
```

> `$rows` = Python 드라이버가 전달한 딕셔너리 리스트  
> `row` = 리스트의 각 원소 (딕셔너리 한 개)

---

### 2. `MATCH` — 기존 노드 찾기

이미 존재하는 노드를 조건으로 검색합니다. 찾지 못하면 그 행은 처리되지 않습니다.

```cypher
MATCH (m:Movie {tmdbId: row.tmdbId})
```

| 요소 | 설명 |
|------|------|
| `(m:Movie)` | `Movie` 레이블을 가진 노드, 변수명 `m` |
| `{tmdbId: row.tmdbId}` | 조건: `tmdbId` 속성이 일치하는 노드만 |

> **MATCH vs MERGE 차이**: `MATCH`는 찾기만, `MERGE`는 없으면 생성까지

---

### 3. `MERGE` — 없으면 생성, 있으면 재사용

`MATCH` + `CREATE`를 합친 구문. 중복 노드/관계 생성을 방지합니다.

```cypher
-- 노드 MERGE
MERGE (g:Genre {genre_id: row.genre_id})

-- 관계 MERGE
MERGE (m)-[:HAS_GENRE]->(g)
```

- 조건에 맞는 노드/관계가 **없으면** → 새로 생성 (`CREATE` 동작)
- 조건에 맞는 노드/관계가 **있으면** → 기존 것을 그대로 사용 (`MATCH` 동작)

---

### 4. `ON CREATE SET` — 최초 생성 시에만 속성 설정

`MERGE` 뒤에 붙여, 노드가 **새로 생성될 때**만 속성을 설정합니다.  
이미 존재하는 노드의 속성은 덮어쓰지 않습니다.

```cypher
MERGE (m:Movie {tmdbId: row.tmdbId})
ON CREATE SET
    m.title    = row.title,
    m.overview = row.overview
```

---

### 5. `ON MATCH SET` — 이미 존재할 때 속성 갱신

`MERGE` 뒤에 붙여, 노드가 **이미 존재할 때** 속성을 업데이트합니다.

```cypher
MERGE (m:Movie {tmdbId: row.tmdbId})
ON CREATE SET m.title = row.title
ON MATCH SET  m.updated_at = timestamp()
```

> 이 튜토리얼에서는 중복 로드를 방지하기 위해 `ON CREATE SET`만 사용합니다.

---

### 6. `SET` — 속성 추가/덮어쓰기

`MATCH`로 찾은 노드에 무조건 속성을 설정합니다. 이미 있으면 덮어씁니다.

```cypher
MATCH (m:Movie {tmdbId: row.tmdbId})
SET m.keywords = row.keywords,
    m.movieId  = row.movieId
```

> `keywords`·`movieId` 처럼 다른 파일에서 추가로 업데이트할 때 사용

---

### 7. `CREATE CONSTRAINT` — 유니크 제약 조건

특정 속성 값의 중복을 DB 수준에서 방지합니다. `MERGE` 성능도 향상됩니다.

```cypher
CREATE CONSTRAINT unique_tmdb_id IF NOT EXISTS
FOR (m:Movie) REQUIRE m.tmdbId IS UNIQUE;
```

| 절 | 의미 |
|----|------|
| `IF NOT EXISTS` | 이미 있으면 에러 없이 건너뜀 |
| `FOR (m:Movie)` | `Movie` 레이블 노드에 적용 |
| `REQUIRE m.tmdbId IS UNIQUE` | `tmdbId` 속성 값 중복 불가 |

---

### 8. `CREATE INDEX` — 인덱스 생성

특정 속성으로 자주 검색할 때 속도를 높입니다.

```cypher
CREATE INDEX actor_id IF NOT EXISTS FOR (p:Person) ON (p.actor_id);
```

---

### 9. `MATCH (n) DETACH DELETE n` — 노드와 관계 모두 삭제

노드에 연결된 모든 관계까지 함께 삭제합니다.  
`DELETE`만 쓰면 관계가 남아 있어 오류 발생.

```cypher
MATCH (n) DETACH DELETE n   -- DB 전체 초기화
```

---

### 10. 관계(Relationship) 표현

```cypher
(a)-[:ACTED_IN]->(m)    -- a가 m에 출연
(d)-[:DIRECTED]->(m)    -- d가 m을 감독
(m)-[:HAS_GENRE]->(g)   -- m이 장르 g를 가짐
```

| 방향 | 표기 |
|------|------|
| 방향 있는 관계 | `(a)-[:REL]->(b)` |
| 관계에 속성 추가 | `MERGE (p)-[r:RATED]->(m) ON CREATE SET r.rating = row.rating` |

---
## Step 4. 그래프 인스턴스 생성

> **DriverError 방지**: `CreateGraph`는 Step 2에서 만든 `driver`를 그대로 받아 씁니다.  
> 드라이버가 두 개 생기지 않으므로 `DriverError: Driver closed` 문제가 발생하지 않습니다.

In [79]:
# Step 2에서 만든 driver를 그대로 전달 — 연결을 새로 열지 않습니다.
graph = CreateGraph(driver)
print("CreateGraph 인스턴스 생성 완료")

CreateGraph 인스턴스 생성 완료


---
## Step 5. 데이터베이스 초기화

> **주의**: 이 셀을 실행하면 Neo4j 데이터베이스의 **모든 데이터가 삭제**됩니다.  
> 새로 구축하는 경우에만 실행하세요.

**사용 Cypher 구문: `MATCH … DETACH DELETE`**

```cypher
MATCH (n) DETACH DELETE n
```

| 구문 | 역할 |
|------|------|
| `MATCH (n)` | 레이블 조건 없이 **모든** 노드를 변수 `n`으로 선택 |
| `DETACH DELETE n` | `n`에 연결된 **모든 관계를 먼저 끊고** 노드를 삭제 |

> `DELETE`만 사용하면 관계가 남아있어 `ConstraintViolationException` 발생.  
> 관계가 있는 노드는 반드시 `DETACH DELETE`를 써야 합니다.

---
## Step 6. 제약 조건 및 인덱스 생성

데이터를 로드하기 전에 유니크 제약 조건과 인덱스를 먼저 설정합니다.  
이렇게 하면 `MERGE` 성능이 크게 향상됩니다.

**사용 Cypher 구문: `CREATE CONSTRAINT`, `CREATE INDEX`**

```cypher
-- 유니크 제약 조건: 같은 tmdbId를 가진 Movie 노드 중복 생성 방지
CREATE CONSTRAINT unique_tmdb_id IF NOT EXISTS
FOR (m:Movie) REQUIRE m.tmdbId IS UNIQUE;

-- 인덱스: actor_id로 Person 노드를 빠르게 검색
CREATE INDEX actor_id IF NOT EXISTS FOR (p:Person) ON (p.actor_id);
```

| 구문 | 역할 |
|------|------|
| `IF NOT EXISTS` | 이미 존재하면 에러 없이 건너뜀 (멱등성 보장) |
| `FOR (m:Movie)` | 적용 대상 레이블 지정 |
| `REQUIRE … IS UNIQUE` | 해당 속성의 유일성 강제 |
| `ON (p.actor_id)` | 인덱스를 생성할 속성 지정 |

> **제약 조건은 자동으로 인덱스도 생성**합니다.  
> `MERGE` 시 Neo4j는 인덱스를 사용해 기존 노드를 O(1)으로 찾기 때문에  
> 대용량 로드 전 반드시 먼저 실행해야 합니다.

| 제약 조건 / 인덱스 | 대상 노드 | 속성 |
|---|---|---|
| UNIQUE | Movie | tmdbId |
| UNIQUE | Movie | movieId |
| UNIQUE | ProductionCompany | company_id |
| UNIQUE | Genre | genre_id |
| UNIQUE | SpokenLanguage | language_code |
| UNIQUE | Country | country_code |
| INDEX | Person | actor_id |
| INDEX | Person | crew_id |
| INDEX | Movie | movieId |
| INDEX | Person | user_id |

---
## Step 7. 영화(Movie) 노드 로드

전체 45,572개 영화 중 `MOVIE_LIMIT`개만 로드합니다.  
이후 모든 관계 데이터는 이 영화들만 대상으로 연결됩니다.

> **데이터 경로**: 노트북과 같은 디렉터리의 `normalized_data/` 폴더를 직접 읽습니다.  
> pandas가 로컬 CSV를 읽고, `UNWIND`로 Neo4j에 전달하므로 `file:///` URI나 APOC가 필요 없습니다.

**사용 Cypher 구문: `UNWIND`, `MERGE`, `ON CREATE SET`**

```cypher
UNWIND $rows AS row
MERGE (m:Movie {tmdbId: row.tmdbId})
ON CREATE SET
    m.title    = row.title,
    m.overview = row.overview,
    m.budget   = row.budget
    -- ... (나머지 속성 생략)
```

| 구문 | 역할 |
|------|------|
| `UNWIND $rows AS row` | Python 리스트 `$rows`의 각 딕셔너리를 `row`로 바인딩 |
| `MERGE (m:Movie {tmdbId: row.tmdbId})` | `tmdbId`로 Movie 노드를 찾거나 없으면 새로 생성 |
| `ON CREATE SET m.title = row.title` | **새로 생성될 때만** `title` 속성 설정 (기존 노드는 유지) |

> `MERGE`의 조건 속성(`tmdbId`)에 **유니크 제약 조건**이 있으면  
> Neo4j가 인덱스로 O(1) 조회하여 대용량 처리에도 빠릅니다.

**로드되는 Movie 속성:**
- `tmdbId`, `title`, `original_title`, `adult`
- `budget`, `revenue`, `runtime`, `release_date`
- `original_language`, `tagline`, `overview`
- `belongs_to_collection`

In [80]:
DATA_DIR = "normalized_data"
MOVIE_LIMIT = 12000  # 전체 45,572개 중 12,000개만 로드

graph.load_movies(f"{DATA_DIR}/normalized_movies.csv", MOVIE_LIMIT)

영화 로드 완료 (12,000개): normalized_data/normalized_movies.csv


로드된 영화 수를 확인합니다.

---
## Step 8. 장르(Genre) 노드 & HAS_GENRE 관계 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE` (노드), `MERGE` (관계), `ON CREATE SET`**

```cypher
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})          -- ① 이미 존재하는 Movie 찾기
MERGE (g:Genre {genre_id: row.genre_id})      -- ② Genre 노드 찾거나 생성
  ON CREATE SET g.genre_name = row.genre_name -- ③ 새 Genre일 때만 이름 설정
MERGE (m)-[:HAS_GENRE]->(g)                   -- ④ Movie→Genre 관계 생성
```

| 단계 | 구문 | 역할 |
|------|------|------|
| ① | `MATCH (m:Movie {…})` | 앞서 로드한 Movie만 처리, 없으면 해당 행 건너뜀 |
| ② | `MERGE (g:Genre {genre_id: …})` | 장르 노드를 중복 없이 생성 |
| ③ | `ON CREATE SET g.genre_name` | Genre가 처음 만들어질 때만 이름 속성 설정 |
| ④ | `MERGE (m)-[:HAS_GENRE]->(g)` | Movie와 Genre 사이 방향 있는 관계를 중복 없이 생성 |

> **MATCH를 먼저 쓰는 이유**: Movie가 없는 행(MOVIE_LIMIT 초과 영화)은  
> 자동으로 제외되어 고아(orphan) 장르 노드가 생기지 않습니다.

In [81]:
graph.load_genres(f"{DATA_DIR}/normalized_genres.csv")

장르 로드 완료 (91,094행): normalized_data/normalized_genres.csv


---
## Step 9. 제작사(ProductionCompany) 노드 & PRODUCED_BY 관계 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE`, `ON CREATE SET` (Step 8과 동일 패턴)**

```cypher
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
MERGE (pc:ProductionCompany {company_id: row.company_id})
  ON CREATE SET pc.company_name = row.company_name
MERGE (m)-[:PRODUCED_BY]->(pc)
```

> Step 8(장르)과 동일한 `MATCH → MERGE 노드 → MERGE 관계` 패턴입니다.  
> 하나의 제작사가 여러 영화에 연결되어도 `ProductionCompany` 노드는 한 개만 생성됩니다.

In [82]:
graph.load_production_companies(f"{DATA_DIR}/normalized_production_companies.csv")

제작사 로드 완료 (70,545행): normalized_data/normalized_production_companies.csv


---
## Step 10. 제작 국가(Country) 노드 & PRODUCED_IN 관계 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE`, `ON CREATE SET`**

```cypher
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
MERGE (c:Country {country_code: row.country_code})   -- 문자열 키('US', 'KR' 등)
  ON CREATE SET c.country_name = row.country_name
MERGE (m)-[:PRODUCED_IN]->(c)
```

> `country_code`는 정수가 아닌 **문자열**('US', 'KR' 등)로 MERGE 조건을 사용합니다.

In [83]:
graph.load_production_countries(f"{DATA_DIR}/normalized_production_countries.csv")

제작 국가 로드 완료 (49,420행): normalized_data/normalized_production_countries.csv


---
## Step 11. 사용 언어(SpokenLanguage) 노드 & HAS_LANGUAGE 관계 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE`, `ON CREATE SET`**

```cypher
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
MERGE (l:SpokenLanguage {language_code: row.language_code})  -- 'en', 'ko' 등
  ON CREATE SET l.language_name = row.language_name
MERGE (m)-[:HAS_LANGUAGE]->(l)
```

> Steps 8–11 모두 동일한 패턴: `MATCH Movie → MERGE 새 노드 → MERGE 관계`  
> 이 패턴이 Neo4j 그래프 구축의 핵심 관용구입니다.

In [84]:
graph.load_spoken_languages(f"{DATA_DIR}/normalized_spoken_languages.csv")

사용 언어 로드 완료 (53,300행): normalized_data/normalized_spoken_languages.csv


---
## Step 12. 키워드(keywords) 속성 로드

키워드는 별도 노드를 만들지 않고 Movie 노드의 `keywords` 속성으로 저장됩니다.  
이후 임베딩 생성 시 텍스트 컨텍스트로 활용됩니다.

**사용 Cypher 구문: `UNWIND`, `MATCH`, `SET`**

```cypher
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
SET m.keywords = row.keywords
```

| 구문 | 역할 |
|------|------|
| `MATCH (m:Movie {…})` | 기존 Movie 노드를 찾음 |
| `SET m.keywords = …` | 속성을 **무조건 덮어씁니다** (`ON CREATE SET`과 달리) |

> **`SET` vs `ON CREATE SET`**  
> - `SET`: 노드가 이미 있어도 속성을 업데이트  
> - `ON CREATE SET`: 노드가 새로 생성될 때만 속성 설정  
> 키워드는 별도 파일에서 기존 Movie 노드에 추가하는 것이므로 `SET`을 사용합니다.

In [85]:
graph.load_keywords(f"{DATA_DIR}/normalized_keywords.csv")

키워드 로드 완료 (45,432행): normalized_data/normalized_keywords.csv


---
## Step 13. 배우(Person/Actor) 노드 & ACTED_IN 관계 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE` (노드), `MERGE` (관계 + 속성), `ON CREATE SET`, `SET` (레이블 추가)**

```cypher
-- ① 배우 노드와 ACTED_IN 관계 생성
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
MERGE (p:Person {actor_id: row.actor_id})
  ON CREATE SET p.name = row.name, p.role = 'actor'
MERGE (p)-[a:ACTED_IN]->(m)
  ON CREATE SET a.character = row.character,   -- 관계에도 속성을 설정할 수 있음
               a.cast_id   = row.cast_id

-- ② Actor 레이블 추가 (별도 쿼리)
MATCH (n:Person) WHERE n.role = 'actor'
SET n:Actor
```

| 구문 | 역할 |
|------|------|
| `MERGE (p:Person {actor_id: …})` | `actor_id`로 Person 노드를 중복 없이 생성 |
| `MERGE (p)-[a:ACTED_IN]->(m)` | 관계를 변수 `a`로 참조, `ON CREATE SET`으로 관계 속성 추가 |
| `SET n:Actor` | 기존 노드에 **추가 레이블** 부여 (`Person` 레이블은 유지됨) |

> **멀티 레이블**: Neo4j 노드는 여러 레이블을 동시에 가질 수 있습니다.  
> `Person` & `Actor` 두 레이블을 모두 가진 노드로 `MATCH (p:Actor)` 또는 `MATCH (p:Person)`으로 검색 가능합니다.

> **대용량 처리**: cast CSV는 55만 행이므로 Python에서 `chunksize=20000`으로 청크 단위 처리합니다.

In [86]:
graph.load_person_actors(f"{DATA_DIR}/normalized_cast.csv")

배우 로드 완료 (559,733행): normalized_data/normalized_cast.csv
Actor 레이블 추가 완료


---
## Step 14. 크루(감독/프로듀서) 노드 & DIRECTED/PRODUCED 관계 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE`, `ON CREATE SET` — job별 쿼리 분리**

원래 `graph_build.py`는 APOC의 `apoc.create.relationship()`으로 관계 타입을 동적으로 생성했지만,  
이 노트북에서는 Python에서 `job` 컬럼으로 필터링한 뒤 **관계 타입별로 쿼리를 분리**합니다.

```cypher
-- 감독 쿼리 (Directors)
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
MERGE (p:Person {crew_id: row.crew_id})
  ON CREATE SET p.name = row.name, p.role = 'Director'
MERGE (p)-[:DIRECTED]->(m)          -- 고정 관계 타입

-- 프로듀서 쿼리 (Producers)
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
MERGE (p:Person {crew_id: row.crew_id})
  ON CREATE SET p.name = row.name, p.role = 'Producer'
MERGE (p)-[:PRODUCED]->(m)          -- 고정 관계 타입

-- 레이블 추가
MATCH (n:Person) WHERE n.role = 'Director' SET n:Director
MATCH (n:Person) WHERE n.role = 'Producer' SET n:Producer
```

| 설계 결정 | 이유 |
|-----------|------|
| job별로 쿼리 분리 | Cypher는 관계 타입을 런타임 변수로 지정 불가 — 타입은 리터럴이어야 함 |
| Python에서 DataFrame 필터링 | `df[df['job'] == 'Director']`로 분리 후 각 쿼리에 전달 |
| APOC 제거 | 외부 플러그인 없이 동일한 결과 달성 |

> **Cypher 관계 타입은 변수를 사용할 수 없습니다.**  
> `MERGE (p)-[:{rel_type}]->(m)` 처럼 동적 타입을 쓰려면 APOC(`apoc.create.relationship`)이 필요합니다.  
> 이 튜토리얼은 APOC 없이 Python 레벨에서 분기 처리합니다.

In [87]:
graph.load_person_crew(f"{DATA_DIR}/normalized_crew.csv")

크루 로드 완료 (감독 19,895, 프로듀서 20,177): normalized_data/normalized_crew.csv


---
## Step 15. 링크(movieId/imdbId) 속성 로드

Movie 노드에 MovieLens `movieId`와 IMDb `imdbId`를 추가합니다.  
이 속성들은 이후 평점 데이터 연결에 사용됩니다.

**사용 Cypher 구문: `UNWIND`, `MATCH`, `SET` (다중 속성 동시 설정)**

```cypher
UNWIND $rows AS row
MATCH (m:Movie {tmdbId: row.tmdbId})
SET m.movieId = row.movieId,     -- 쉼표로 여러 속성을 한 번에 설정
    m.imdbId  = row.imdbId
```

| 구문 | 역할 |
|------|------|
| `SET m.movieId = …, m.imdbId = …` | 쉼표로 구분하여 여러 속성을 동시에 업데이트 |

> `movieId`는 이후 `load_ratings`에서 `MATCH (m:Movie {movieId: …})`로 평점과 연결하는 **조인 키**입니다.  
> `links` 로드 전에 평점을 먼저 로드하면 매칭이 되지 않으므로 순서가 중요합니다.

In [88]:
graph.load_links(f"{DATA_DIR}/normalized_links.csv")

링크 로드 완료 (45,624행): normalized_data/normalized_links.csv


---
## Step 16. 사용자 평점(User/RATED) 로드

**사용 Cypher 구문: `UNWIND`, `MATCH`, `MERGE` (노드), `MERGE` (관계 + 속성), `ON CREATE SET`, `SET` (레이블)**

```cypher
-- ① 평점 노드와 관계 생성
UNWIND $rows AS row
MATCH (m:Movie {movieId: row.movieId})     -- tmdbId가 아닌 movieId로 MATCH!
MERGE (p:Person {user_id: row.userId})
  ON CREATE SET p.role = 'user'
MERGE (p)-[r:RATED]->(m)
  ON CREATE SET r.rating    = row.rating,   -- 관계에 평점 속성
               r.timestamp = row.timestamp  -- 관계에 타임스탬프 속성

-- ② User 레이블 추가
MATCH (n:Person) WHERE n.role = 'user'
SET n:User
```

| 구문 | 역할 |
|------|------|
| `MATCH (m:Movie {movieId: …})` | `tmdbId` 대신 `movieId`(Step 15에서 추가)로 영화 조회 |
| `MERGE (p:Person {user_id: …})` | 사용자 노드 — 이름 없이 `user_id`만 가짐 |
| `MERGE (p)-[r:RATED]->(m)` ON CREATE SET | 평점 관계에 `rating`, `timestamp` 속성 저장 |

> **관계에 속성을 저장하는 이유**:  
> `rating`과 `timestamp`는 "User가 Movie를 평가한 행위"에 속하는 데이터이므로  
> 노드가 아닌 `RATED` **관계**의 속성으로 저장합니다. 이것이 그래프 모델링의 핵심입니다.

In [89]:
graph.load_ratings(f"{DATA_DIR}/normalized_ratings_small.csv")

평점 로드 완료 (100,004행): normalized_data/normalized_ratings_small.csv
User 레이블 추가 완료


In [90]:
# 평점 분포 확인
with driver.session() as session:
    result = session.run("""
        MATCH ()-[r:RATED]->()
        RETURN r.rating AS rating, count(*) AS cnt
        ORDER BY rating
    """)
    print("평점 분포:")
    for row in result:
        bar = "█" * (row['cnt'] // 2000)
        print(f"  {row['rating']:.1f}점  {bar} ({row['cnt']:,}건)")

평점 분포:
  0.5점   (920건)
  1.0점  █ (3,084건)
  1.5점   (1,342건)
  2.0점  ███ (6,690건)
  2.5점  █ (3,695건)
  3.0점  █████████ (18,784건)
  3.5점  ████ (8,869건)
  4.0점  █████████████ (26,385건)
  4.5점  ███ (6,524건)
  5.0점  ███████ (14,051건)


---
## Step 17. 그래프 검증

전체 노드 수와 관계 수를 확인합니다.

In [91]:
def fetch_node_counts(session):
    labels = [r["label"] for r in session.run("CALL db.labels() YIELD label RETURN label")]
    print("\n📦 노드 수:")
    for label in sorted(labels):
        count = session.run(f"MATCH (n:`{label}`) RETURN count(n) AS count").single()["count"]
        print(f"  {label:25s}: {count:>10,}")

def fetch_relationship_counts(session):
    types = [r["relationshipType"] for r in session.run(
        "CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType"
    )]
    print("\n🔗 관계 수:")
    for rel in sorted(types):
        count = session.run(f"MATCH ()-[:`{rel}`]->() RETURN count(*) AS count").single()["count"]
        print(f"  {rel:25s}: {count:>10,}")

with driver.session() as session:
    fetch_node_counts(session)
    fetch_relationship_counts(session)


📦 노드 수:
  Actor                    :     81,165
  Country                  :        112
  Director                 :      5,047
  Genre                    :         20
  Movie                    :     11,997
  Person                   :     92,663
  Producer                 :      5,780
  ProductionCompany        :      7,961
  SpokenLanguage           :        100
  User                     :        671

🔗 관계 수:
  ACTED_IN                 :    191,307
  DIRECTED                 :      5,047
  HAS_GENRE                :     28,479
  HAS_LANGUAGE             :     16,184
  PRODUCED                 :      6,939
  PRODUCED_BY              :     22,758
  PRODUCED_IN              :     14,700
  RATED                    :     90,344


---
## Step 18. 그래프 탐색 예제 쿼리

아래는 구축된 그래프를 활용한 다양한 Cypher 쿼리 예제입니다.

### 18-1. 특정 배우가 출연한 액션 영화

In [92]:
with driver.session() as session:
    result = session.run("""
        MATCH (a:Actor)-[:ACTED_IN]->(m:Movie)-[:HAS_GENRE]->(g:Genre)
        WHERE a.name = 'Tom Hanks' AND g.genre_name = 'Drama'
        RETURN m.title AS title, m.release_date AS released
        ORDER BY released DESC LIMIT 10
    """)
    print("Tom Hanks 출연 드라마:")
    for r in result:
        print(f"  {r['title']} ({r['released']})")

Tom Hanks 출연 드라마:
  The Terminal (2004-06-17)
  Catch Me If You Can (2002-12-25)
  Road to Perdition (2002-07-12)
  Cast Away (2000-12-22)
  The Green Mile (1999-12-10)
  Saving Private Ryan (1998-07-24)
  From the Earth to the Moon (1998-04-05)
  That Thing You Do! (1996-10-04)
  Apollo 13 (1995-06-30)
  Forrest Gump (1994-07-06)


### 18-2. 두 배우가 함께 출연한 영화 (협업 그래프)

In [93]:
with driver.session() as session:
    result = session.run("""
        MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
        WHERE a1.name = 'Tom Hanks' AND a2.name = 'Meg Ryan'
        RETURN m.title AS title, m.release_date AS released
        ORDER BY released
    """)
    print("Tom Hanks & Meg Ryan 공동 출연작:")
    rows = list(result)
    if rows:
        for r in rows:
            print(f"  {r['title']} ({r['released']})")
    else:
        print("  해당 배우 조합의 영화가 없습니다.")

Tom Hanks & Meg Ryan 공동 출연작:
  Joe Versus the Volcano (1990-03-09)
  Sleepless in Seattle (1993-06-24)
  You've Got Mail (1998-12-17)


### 18-3. 평균 평점이 가장 높은 영화 Top 10

In [94]:
with driver.session() as session:
    result = session.run("""
        MATCH (u:User)-[r:RATED]->(m:Movie)
        WITH m, avg(r.rating) AS avg_rating, count(r) AS num_ratings
        WHERE num_ratings >= 50
        RETURN m.title AS title, round(avg_rating, 2) AS avg_rating, num_ratings
        ORDER BY avg_rating DESC LIMIT 10
    """)
    print("평균 평점 Top 10 (50건 이상 평가된 영화):")
    for r in result:
        print(f"  {r['title']:40s} ⭐ {r['avg_rating']} ({r['num_ratings']:,}명)")

평균 평점 Top 10 (50건 이상 평가된 영화):
  The Shawshank Redemption                 ⭐ 4.49 (311명)
  The Godfather                            ⭐ 4.49 (200명)
  The African Queen                        ⭐ 4.42 (50명)
  The Godfather: Part II                   ⭐ 4.39 (135명)
  The Maltese Falcon                       ⭐ 4.39 (62명)
  The Usual Suspects                       ⭐ 4.37 (201명)
  Raging Bull                              ⭐ 4.35 (50명)
  Chinatown                                ⭐ 4.34 (76명)
  Rear Window                              ⭐ 4.32 (92명)
  Schindler's List                         ⭐ 4.3 (244명)


### 18-4. 특정 감독의 장르 분포

In [95]:
with driver.session() as session:
    result = session.run("""
        MATCH (d:Director)-[:DIRECTED]->(m:Movie)-[:HAS_GENRE]->(g:Genre)
        WHERE d.name = 'Steven Spielberg'
        RETURN g.genre_name AS genre, count(m) AS cnt
        ORDER BY cnt DESC
    """)
    rows = list(result)
    if rows:
        print("Steven Spielberg 감독 장르 분포:")
        for r in rows:
            print(f"  {r['genre']:20s} {r['cnt']}편")
    else:
        print("해당 감독 데이터 없음 (로드된 영화에 포함되지 않았을 수 있습니다)")

Steven Spielberg 감독 장르 분포:
  Adventure            1편
  Science Fiction      1편


### 18-5. 멀티홉 추천 — 같은 장르를 좋아하는 유저가 높게 평가한 영화

In [96]:
with driver.session() as session:
    result = session.run("""
        // 'The Dark Knight'를 4점 이상 준 유저
        MATCH (u:User)-[r1:RATED]->(seed:Movie)
        WHERE seed.title CONTAINS 'Dark Knight' AND r1.rating >= 4
        // 그 유저가 높게 평가한 다른 영화
        MATCH (u)-[r2:RATED]->(rec:Movie)
        WHERE rec <> seed AND r2.rating >= 4
        WITH rec, count(DISTINCT u) AS supporters
        RETURN rec.title AS title, supporters
        ORDER BY supporters DESC LIMIT 10
    """)
    print("'The Dark Knight' 팬들이 좋아한 영화:")
    for r in result:
        print(f"  {r['title']:40s} {r['supporters']:,}명")

'The Dark Knight' 팬들이 좋아한 영화:


In [97]:
# driver 하나만 닫으면 됩니다. graph.driver는 같은 객체를 가리킵니다.
driver.close()
print("Neo4j 연결이 정상적으로 종료되었습니다.")

Neo4j 연결이 정상적으로 종료되었습니다.


---
## 완료! 다음 단계

그래프 데이터베이스 구축이 완료되었습니다. 다음 단계로 진행하세요:

1. **임베딩 생성**: `generate_embeddings.py` 실행
   - Vertex AI `text-embedding-005` 모델로 Movie 노드의 텍스트 임베딩 생성
   - 또는 `load_embeddings.py`로 사전 계산된 임베딩 로드

2. **추천 챗봇 실행**: `app.py` 실행
   - Gradio UI 기반 GraphRAG 영화 추천 시스템
   - `http://0.0.0.0:8080` 에서 접근

```bash
# 임베딩 생성 (Vertex AI 필요)
python generate_embeddings.py

# 또는 사전 계산된 임베딩 로드
python load_embeddings.py

# 추천 챗봇 실행
python app.py
```